In [10]:
# useful libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [11]:
# setting up default plotting parameters
%matplotlib inline

plt.rcParams['figure.figsize'] = [20.0, 7.0]
plt.rcParams.update({'font.size': 22,})

sns.set_palette('viridis')
sns.set_style('white')
sns.set_context('talk', font_scale=0.8)

In [12]:
# read in data
df = pd.read_csv('train.csv')

print(df.shape)
df.head()

(967332, 14)


,auctionId,timeStamp,placementId,websiteId,hashedRefererDeepThree,country,opeartingSystem,browser,browserVersion,device,environmentType,integrationType,articleSafenessCategorization,isSold
0,001ed16b-dd08-4599-b8ef-4f56a373c454_6e5f1087-...,1603815466,120706,68203,1ae7c2d3c28b711c072d8e2eb3869fa59090669bdc153e...,US,Windows,Chrome,86_0,PC,js-web,2,safe,False
1,0024b36a-4fb5-4070-88fb-fc0bfb1909ed,1603974586,69454,42543,df1108bf6ae49dbccf5eab60ff9d04a6a09dda60ec7290...,RO,Android,Facebook App,293_0,Phone,js-fbwv,1,unsafe,False
2,003630fa-ad63-4283-be1b-141670132d70_f37c2b23-...,1604229969,100170,57703,cc6957e8aec85a4d920991c53874c5d0780bbfbd469802...,UK,Android,Facebook App,294_0,Phone,js-web,2,safe,True
3,0048c65a-ce76-43ba-98d2-8e87607468f8,1604156610,100446,57797,7fc0bb7a65d074e003cce786cda2b070f80dd47179c4b9...,ES,Android,Chrome Mobile,86_0,Phone,js-ampsf,1,safe,True
4,0056b8a7-54f9-4ac8-8d50-f725bf377872,1604004493,119517,67613,3a6552ccbf66ad166aa9005c3e08f70716abd676cfd87b...,FR,Android,Facebook App,293_0,Phone,js-fbwv,1,unsafe,False


In [13]:
print(df.isSold.value_counts())

df = df.rename(columns={'opeartingSystem': 'operatingSystem'})

isSold
False    533425
True     433907
Name: count, dtype: int64


In [14]:
df.set_index('auctionId', inplace = True, drop = True)

In [15]:
# Data preprocessing

X = df.copy()

class_dummies = pd.get_dummies(X['country'], prefix = 'country')
X = X.join(class_dummies)

class_dummies = pd.get_dummies(X['operatingSystem'], prefix = 'operatingSystem')
X = X.join(class_dummies)

class_dummies = pd.get_dummies(X['browser'], prefix = 'browser')
X = X.join(class_dummies)

class_dummies = pd.get_dummies(X['device'], prefix = 'device')
X = X.join(class_dummies)

class_dummies = pd.get_dummies(X['environmentType'], prefix = 'environmentType')
X = X.join(class_dummies)

class_dummies = pd.get_dummies(X['articleSafenessCategorization'], prefix = 'articleSafenessCategorization')
X = X.join(class_dummies)

X = X.drop(columns=['country', 'operatingSystem', 'browser', 'device', 'environmentType', 'articleSafenessCategorization', 'hashedRefererDeepThree', 'browserVersion', 'isSold'])

In [16]:
from sklearn.model_selection import train_test_split

y = df['isSold']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# 1. Définition de la grille d'hyperparamètres à tester
# Tu peux ajuster ces valeurs selon la taille de ton dataset et la puissance de ta machine
param_grid = {
    'n_estimators': [50, 100, 200],          # Nombre d'arbres (10 c'était très bas !)
    'max_depth': [None, 10, 20, 30],         # Profondeur maximale de l'arbre
    'min_samples_split': [2, 5, 10],         # Nombre min de données pour diviser un nœud
    'min_samples_leaf': [1, 2, 4],           # Nombre min de données dans une feuille
    'class_weight': [None, 'balanced']       # Très utile pour booster le F1-score si tes classes sont déséquilibrées
}

# 2. Initialisation du modèle de base
rf_base = RandomForestClassifier(random_state=42)

# 3. Configuration du Grid Search
# scoring='f1' indique qu'on veut optimiser le F1-score
# cv=5 fait une validation croisée à 5 plis (folds)
# n_jobs=-1 utilise tous les cœurs de ton processeur pour aller plus vite
grid_search = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    scoring='f1', 
    cv=5, 
    n_jobs=-1, 
    verbose=1
)

# 4. Entraînement de la boucle d'optimisation
print("Lancement de l'optimisation des hyperparamètres...")
grid_search.fit(X_train, y_train)

# 5. Récupération du meilleur modèle
best_rf = grid_search.best_estimator_

print("\n--- Meilleurs Hyperparamètres trouvés ---")
print(grid_search.best_params_)
print(f"Meilleur F1-score en validation croisée : {grid_search.best_score_:.4f}\n")

Lancement de l'optimisation des hyperparamètres...
Fitting 5 folds for each of 216 candidates, totalling 1080 fits


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, recall_score, precision_score

y_pred = best_rf.predict(X_test)

print('--- Performances sur le jeu de Test ---')
print('Accuracy :        ', accuracy_score(y_test, y_pred))
print('Precision score : ', precision_score(y_test, y_pred))
print('F1 score :        ', f1_score(y_test, y_pred))
print('Recall score :    ', recall_score(y_test, y_pred))

print('\nMatrice de confusion :')
print(confusion_matrix(y_test, y_pred))

accuracy:  0.7415941736833672
precision score:  0.7260986679458322
f1 score:  0.7025306287597956
recall score:  0.6804444546900574
